In [77]:
from cnn.core_concepts import ModelAWithoutLinear, ModelBWithoutLinear, ModelB, ModelA
from core_concepts import ModelA, ModelB
import torch
import torchvision
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import numpy as np
import time


torch.manual_seed(12)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


In [64]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform)

In [56]:
# take the first image from the test for testing purposes.
image=train_dataset[0][0].unsqueeze(0)
image.shape

torch.Size([1, 3, 32, 32])

In [57]:
modelAWithout = ModelAWithoutLinear()
modelBWithout = ModelBWithoutLinear()
shift = 12

In [58]:
# Test ModelA (no pooling)
f_original_A = modelAWithout(image)
f_shifted_A = modelAWithout(image_shifted)
f_expected_A = torch.roll(f_original_A, shifts=(shift, shift), dims=(2, 3))
error_A = torch.norm(f_shifted_A - f_expected_A) / torch.norm(f_expected_A)

print(f"ModelA full conv stack error: {error_A.item():.6f}")
print(f"ModelA output shape: {f_original_A.shape}")


ModelA full conv stack error: 0.561472
ModelA output shape: torch.Size([1, 5, 32, 32])


In [59]:
# Test ModelB (with pooling)
f_original_B = modelBWithout(image)
f_shifted_B = modelBWithout(image_shifted)

# Calculate expected shift after 2 pooling operations
# Each MaxPool2d(2,2) divides shift by 2
# 20 -> 10 -> 5 after two poolings
expected_shift = shift // 4  # Two pooling layers
f_expected_B = torch.roll(f_original_B, shifts=(expected_shift, expected_shift), dims=(2, 3))
error_B = torch.norm(f_shifted_B - f_expected_B) / torch.norm(f_expected_B)

print(f"ModelB full conv stack error: {error_B.item():.6f}")
print(f"ModelB output shape: {f_original_B.shape}")

ModelB full conv stack error: 0.499300
ModelB output shape: torch.Size([1, 5, 8, 8])


In [60]:
shifts = [4, 8, 12, 16]  # Avoid shifts that align perfectly with kernel sizes

errors_A = []
errors_B = []

modelA_test = ModelAWithoutLinear()
modelB_test = ModelBWithoutLinear()

for shift in shifts:
    # Test ModelA
    image_shifted = torch.roll(image, shifts=(shift, shift), dims=(2, 3))

    f_original_A = modelA_test(image)
    f_shifted_A = modelA_test(image_shifted)
    f_expected_A = torch.roll(f_original_A, shifts=(shift, shift), dims=(2, 3))
    error_A = torch.norm(f_shifted_A - f_expected_A) / torch.norm(f_expected_A)
    errors_A.append(error_A.item())

    # Test ModelB
    f_original_B = modelB_test(image)
    f_shifted_B = modelB_test(image_shifted)

    # Calculate expected shift after 2 pooling operations
    expected_shift = shift // 4  # Two pooling layers
    f_expected_B = torch.roll(f_original_B, shifts=(expected_shift, expected_shift), dims=(2, 3))
    error_B = torch.norm(f_shifted_B - f_expected_B) / torch.norm(f_expected_B)
    errors_B.append(error_B.item())

    print(f"Shift {shift}: ModelA error = {error_A.item():.6f}, ModelB error = {error_B.item():.6f}")

print(f"\nAverage ModelA error: {np.mean(errors_A):.6f} ± {np.std(errors_A):.6f}")
print(f"Average ModelB error: {np.mean(errors_B):.6f} ± {np.std(errors_B):.6f}")



Shift 4: ModelA error = 0.239828, ModelB error = 0.164942
Shift 8: ModelA error = 0.242571, ModelB error = 0.177042
Shift 12: ModelA error = 0.248532, ModelB error = 0.171278
Shift 16: ModelA error = 0.257421, ModelB error = 0.197489

Average ModelA error: 0.247088 ± 0.006745
Average ModelB error: 0.177688 ± 0.012207


In [61]:
# In theory the pooling should have less equivariance and that seems to be true for some shifts, I think this pheneomenom get's more relevant as the network goes deeper, since previoues test with a only one pooling showed even better equivariance. Also, larger shifts seem to be better handled by the model A where we expect the most equivariance.

In [62]:
# time to test invariance.=

In [67]:
modelA_full = ModelA().to(device)
modelB_full = ModelB().to(device)

In [65]:
batch_size = 32
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f"Dataset loaded: {len(train_dataset)} train samples, {len(test_dataset)} test samples")

Dataset loaded: 50000 train samples, 10000 test samples


In [72]:
def train_model(model, train_loader, num_epochs=5, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    losses = []
    accuracies = []
    times = []

    for epoch in range(num_epochs):
        start_time = time.time()
        running_loss = 0.0
        correct = 0
        total = 0

        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            if i % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

        epoch_time = time.time() - start_time
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total

        losses.append(epoch_loss)
        accuracies.append(epoch_acc)
        times.append(epoch_time)

        print(f'Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%, Time: {epoch_time:.2f}s')

    return losses, accuracies, times

In [75]:
print("Training ModelA...")
modelA_losses, modelA_accs, modelA_times = train_model(modelA_full, train_loader, num_epochs=3)

print("\nTraining ModelB...")
modelB_losses, modelB_accs, modelB_times = train_model(modelB_full, train_loader, num_epochs=3)

print(f"\nTraining completed!")
print(f"ModelA - Final Accuracy: {modelA_accs[-1]:.2f}%, Avg Time per Epoch: {np.mean(modelA_times):.2f}s")
print(f"ModelB  - Final Accuracy: {modelB_accs[-1]:.2f}%, Avg Time per Epoch: {np.mean(modelB_times):.2f}s")

Training ModelA...
Epoch [1/3], Step [1/1563], Loss: 1.7268
Epoch [1/3], Step [101/1563], Loss: 1.7601
Epoch [1/3], Step [201/1563], Loss: 1.9168
Epoch [1/3], Step [301/1563], Loss: 1.9424
Epoch [1/3], Step [401/1563], Loss: 1.9282
Epoch [1/3], Step [501/1563], Loss: 1.6718
Epoch [1/3], Step [601/1563], Loss: 1.7428
Epoch [1/3], Step [701/1563], Loss: 1.7107
Epoch [1/3], Step [801/1563], Loss: 1.9169
Epoch [1/3], Step [901/1563], Loss: 2.0620
Epoch [1/3], Step [1001/1563], Loss: 1.9973
Epoch [1/3], Step [1101/1563], Loss: 1.9237
Epoch [1/3], Step [1201/1563], Loss: 1.8300
Epoch [1/3], Step [1301/1563], Loss: 1.6962
Epoch [1/3], Step [1401/1563], Loss: 1.5694
Epoch [1/3], Step [1501/1563], Loss: 1.9075
Epoch [1/3] - Loss: 1.7974, Accuracy: 30.64%, Time: 11.71s
Epoch [2/3], Step [1/1563], Loss: 1.7034
Epoch [2/3], Step [101/1563], Loss: 1.8385
Epoch [2/3], Step [201/1563], Loss: 1.7256
Epoch [2/3], Step [301/1563], Loss: 1.7334
Epoch [2/3], Step [401/1563], Loss: 1.6426
Epoch [2/3], Step

In [76]:
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

modelA_test_acc = evaluate_model(modelA_full, test_loader)
modelB_test_acc = evaluate_model(modelB_full, test_loader)

print(f"Test Results:")
print(f"ModelA Test Accuracy: {modelA_test_acc:.2f}%")
print(f"ModelB Test Accuracy: {modelB_test_acc:.2f}%")

Test Results:
ModelA Test Accuracy: 31.08%
ModelB CNN Test Accuracy: 35.49%


In [79]:
class ShiftedDataset(torch.utils.data.Dataset):
    def __init__(self, original_dataset, shift):
        self.original_dataset = original_dataset
        self.shift = shift

    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, idx):
        image, label = self.original_dataset[idx]
        # Apply shift to the image
        shifted_image = torch.roll(image, shifts=(self.shift, self.shift), dims=(1, 2))
        return shifted_image, label


# Test invariance with different shifts
def test_invariance_with_shifts(model, test_dataset, shifts, device, batch_size=32):
    model.eval()
    original_acc = evaluate_model(model,
                                  torch.utils.data.DataLoader(test_dataset, batch_size=batch_size))

    print(f"Original accuracy: {original_acc:.2f}%")

    for shift in shifts:
        shifted_dataset = ShiftedDataset(test_dataset, shift)
        shifted_loader = torch.utils.data.DataLoader(shifted_dataset, batch_size=batch_size)
        shifted_acc = evaluate_model(model, shifted_loader)

        accuracy_drop = original_acc - shifted_acc
        print(f"Shift {shift}px: {shifted_acc:.2f}% (drop: {accuracy_drop:.2f}%)")

    return original_acc


# Test both models
shifts = [4, 8, 12, 16]
print("=== ModelA Translation Invariance ===")
modelA_orig_acc = test_invariance_with_shifts(modelA_full, test_dataset, shifts, device)

print("\n=== ModelB Translation Invariance ===")
modelB_orig_acc = test_invariance_with_shifts(modelB_full, test_dataset, shifts, device)

=== ModelA Translation Invariance ===
Original accuracy: 31.08%
Shift 4px: 30.89% (drop: 0.19%)
Shift 8px: 30.74% (drop: 0.34%)
Shift 12px: 30.40% (drop: 0.68%)
Shift 16px: 30.28% (drop: 0.80%)

=== ModelB Translation Invariance ===
Original accuracy: 35.49%
Shift 4px: 32.68% (drop: 2.81%)
Shift 8px: 32.59% (drop: 2.90%)
Shift 12px: 32.25% (drop: 3.24%)
Shift 16px: 32.71% (drop: 2.78%)


In [81]:
shifts = [4, 8, 12, 16]  # Avoid shifts that align perfectly with kernel sizes

errors_A = []
errors_B = []

print("=== ModelA and ModelB (pooling) Trained Translation Equivariance ===")
modelA_trained = modelA_full.model_a
modelB_trained = modelB_full.model_b

for shift in shifts:
    # Test ModelA
    image_shifted = torch.roll(image, shifts=(shift, shift), dims=(2, 3))

    f_original_A = modelA_test(image)
    f_shifted_A = modelA_test(image_shifted)
    f_expected_A = torch.roll(f_original_A, shifts=(shift, shift), dims=(2, 3))
    error_A = torch.norm(f_shifted_A - f_expected_A) / torch.norm(f_expected_A)
    errors_A.append(error_A.item())

    # Test ModelB
    f_original_B = modelB_test(image)
    f_shifted_B = modelB_test(image_shifted)

    # Calculate expected shift after 2 pooling operations
    expected_shift = shift // 4  # Two pooling layers
    f_expected_B = torch.roll(f_original_B, shifts=(expected_shift, expected_shift), dims=(2, 3))
    error_B = torch.norm(f_shifted_B - f_expected_B) / torch.norm(f_expected_B)
    errors_B.append(error_B.item())

    print(f"Shift {shift}: ModelA error = {error_A.item():.6f}, ModelB error = {error_B.item():.6f}")

print(f"\nAverage ModelA error: {np.mean(errors_A):.6f} ± {np.std(errors_A):.6f}")
print(f"Average ModelB error: {np.mean(errors_B):.6f} ± {np.std(errors_B):.6f}")

=== ModelA and ModelB (pooling) Trained Translation Equivariance ===
Shift 4: ModelA error = 0.239828, ModelB error = 0.164942
Shift 8: ModelA error = 0.242571, ModelB error = 0.177042
Shift 12: ModelA error = 0.248532, ModelB error = 0.171278
Shift 16: ModelA error = 0.257421, ModelB error = 0.197489

Average ModelA error: 0.247088 ± 0.006745
Average ModelB error: 0.177688 ± 0.012207


In [ ]:
# Doing the equivariance test on the trained model showed that pooling performs better